# Can we predict next-month account churn?

This notebook tests whether the information available in a monthly account snapshot can rank accounts by their risk of cancelling in the following month. It emphasizes a realistic **out-of-time evaluation** and explicitly removes a future-derived leakage field.

**Decision rule:** the model is useful if it beats a prevalence-only baseline on the untouched final three months, especially on average precision and lift in the highest-risk decile. ROC-AUC measures overall ranking; average precision is particularly informative for the minority churn class.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, log_loss,
    precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 50)
RANDOM_STATE = 42

## 1. Load the data and dictionary

In [ ]:
DATA_PATH = Path('data.csv')
DICTIONARY_PATH = Path('columns.csv')
assert DATA_PATH.exists() and DICTIONARY_PATH.exists(), 'Run this notebook beside data.csv and columns.csv'

df = pd.read_csv(DATA_PATH)
dictionary = pd.read_csv(DICTIONARY_PATH)
df['snapshot_date'] = pd.to_datetime(df['snapshot_month'], format='%Y-%m')
display(dictionary)
display(df.head())

In [ ]:
quality = pd.Series({
    'rows': len(df),
    'accounts': df['account_ref'].nunique(),
    'months': df['snapshot_month'].nunique(),
    'first_month': df['snapshot_month'].min(),
    'last_month': df['snapshot_month'].max(),
    'duplicate_account_months': df.duplicated(['account_ref', 'snapshot_month']).sum(),
    'missing_cells': int(df.isna().sum().sum()),
    'churn_rate': df['churn_next_month'].mean(),
})
display(quality.to_frame('value'))
assert set(df['churn_next_month'].unique()) <= {0, 1}
assert quality['duplicate_account_months'] == 0
assert quality['missing_cells'] == 0

## 2. Leakage audit

`account_ref` is an identifier, so it is not a deployable predictor for new or changing accounts. More importantly, `lifetime_months` was computed from a CRM export dated after every snapshot. It contains future information. The check below shows that `lifetime_months == tenure_months` identifies every positive label exactly, so this column must be excluded.

In [ ]:
leak_rule = (df['lifetime_months'] == df['tenure_months']).astype(int)
leak_audit = pd.crosstab(
    leak_rule.rename('lifetime equals tenure'),
    df['churn_next_month'],
    margins=True,
)
display(leak_audit)
print(f"Invalid leaked-rule ROC-AUC: {roc_auc_score(df['churn_next_month'], leak_rule):.3f}")
assert np.array_equal(leak_rule.to_numpy(), df['churn_next_month'].to_numpy())

## 3. Time pattern and evaluation design

Randomly splitting rows would mix earlier and later snapshots and understate temporal drift. We reserve **2025-09 through 2025-11** as an untouched test period. We use **2025-08** as validation when choosing how much history to train on. This is operationally realistic: train on past snapshots, score future snapshots. Accounts may recur across time, but their identifier is excluded from the features.

In [ ]:
monthly = df.groupby('snapshot_date')['churn_next_month'].agg(['size', 'mean'])
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(monthly.index, monthly['mean'], marker='o', color='tab:red')
ax1.set(ylabel='Churn rate', xlabel='Snapshot month', title='Churn prevalence changes materially over time')
ax1.axvspan(pd.Timestamp('2025-09-01'), pd.Timestamp('2025-11-30'), alpha=.12, color='tab:blue', label='test period')
ax1.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
display(monthly.tail(8).style.format({'mean': '{:.1%}'}))

## 4. Leakage-free features and model

The feature set uses only fields known during the snapshot month. Calendar month is encoded cyclically to represent seasonality without treating dates as arbitrary categories. A regularized logistic regression is a strong, interpretable first model for this dataset. We compare it with a prevalence-only dummy baseline.

In [ ]:
df['calendar_month'] = df['snapshot_date'].dt.month
df['month_sin'] = np.sin(2 * np.pi * df['calendar_month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['calendar_month'] / 12)

numeric_features = [
    'seats', 'monthly_fee', 'tenure_months', 'usage_hours',
    'support_tickets', 'month_sin', 'month_cos',
]
categorical_features = ['plan']
features = categorical_features + numeric_features
target = 'churn_next_month'
excluded = sorted(set(df.columns) - set(features) - {target})
print('Model features:', features)
print('Excluded fields:', excluded)

def make_model(estimator):
    prep = ColumnTransformer([
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ])
    return Pipeline([('preprocess', prep), ('model', estimator)])

def score_predictions(y, probability, name):
    order = np.argsort(-probability)
    top_n = max(1, int(np.ceil(0.10 * len(y))))
    top_rate = y.iloc[order[:top_n]].mean()
    return {
        'model': name,
        'roc_auc': roc_auc_score(y, probability),
        'average_precision': average_precision_score(y, probability),
        'log_loss': log_loss(y, probability),
        'brier_score': brier_score_loss(y, probability),
        'mean_prediction': probability.mean(),
        'actual_rate': y.mean(),
        'top_decile_precision': top_rate,
        'top_decile_lift': top_rate / y.mean(),
    }

### Choose the training window on validation data

Because the chart shows drift, we compare several expanding/recent windows. We select the window with the highest validation average precision, then refit it through August before looking at the test set.

In [ ]:
validation = df[df['snapshot_month'].eq('2025-08')].copy()
window_starts = ['2024-01', '2024-09', '2025-01', '2025-03', '2025-05']
validation_rows = []

for start in window_starts:
    train = df[(df['snapshot_month'] >= start) & (df['snapshot_month'] < '2025-08')]
    candidate = make_model(LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE))
    candidate.fit(train[features], train[target])
    p = candidate.predict_proba(validation[features])[:, 1]
    row = score_predictions(validation[target], p, f'train from {start}')
    row['training_rows'] = len(train)
    row['window_start'] = start
    validation_rows.append(row)

validation_results = pd.DataFrame(validation_rows).sort_values('average_precision', ascending=False)
display(validation_results.set_index('model').style.format({
    'roc_auc': '{:.3f}', 'average_precision': '{:.3f}', 'log_loss': '{:.3f}',
    'brier_score': '{:.3f}', 'mean_prediction': '{:.1%}', 'actual_rate': '{:.1%}',
    'top_decile_precision': '{:.1%}', 'top_decile_lift': '{:.2f}x',
}))
selected_start = validation_results.iloc[0]['window_start']
print('Selected window start:', selected_start)

## 5. Final untouched test evaluation

In [ ]:
train_final = df[(df['snapshot_month'] >= selected_start) & (df['snapshot_month'] < '2025-09')].copy()
test = df[df['snapshot_month'] >= '2025-09'].copy()

models = {
    'Prevalence baseline': make_model(DummyClassifier(strategy='prior')),
    'Logistic regression': make_model(LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE)),
}
test_rows, probabilities = [], {}
for name, model in models.items():
    model.fit(train_final[features], train_final[target])
    p = model.predict_proba(test[features])[:, 1]
    probabilities[name] = p
    test_rows.append(score_predictions(test[target], p, name))

test_results = pd.DataFrame(test_rows).set_index('model')
display(test_results.style.format({
    'roc_auc': '{:.3f}', 'average_precision': '{:.3f}', 'log_loss': '{:.3f}',
    'brier_score': '{:.3f}', 'mean_prediction': '{:.1%}', 'actual_rate': '{:.1%}',
    'top_decile_precision': '{:.1%}', 'top_decile_lift': '{:.2f}x',
}))
print(f'Train: {train_final.snapshot_month.min()} to {train_final.snapshot_month.max()} ({len(train_final):,} rows)')
print(f'Test:  {test.snapshot_month.min()} to {test.snapshot_month.max()} ({len(test):,} rows)')

In [ ]:
y_test = test[target]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for name, p in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    precision, recall, _ = precision_recall_curve(y_test, p)
    axes[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_test, p):.3f})")
    axes[1].plot(recall, precision, label=f"{name} ({average_precision_score(y_test, p):.3f})")
axes[0].plot([0, 1], [0, 1], '--', color='gray')
axes[0].set(xlabel='False-positive rate', ylabel='True-positive rate', title='ROC curve')
axes[1].axhline(y_test.mean(), ls='--', color='gray', label=f'Prevalence ({y_test.mean():.3f})')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision–recall curve')
for ax in axes: ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. What drives the ranking?

Permutation importance measures the drop in test average precision when each raw input is shuffled. It is descriptive, not causal.

In [ ]:
final_model = models['Logistic regression']
importance = permutation_importance(
    final_model, test[features], y_test, scoring='average_precision',
    n_repeats=20, random_state=RANDOM_STATE, n_jobs=1,
)
importance_df = pd.DataFrame({
    'feature': features,
    'mean_ap_drop': importance.importances_mean,
    'std': importance.importances_std,
}).sort_values('mean_ap_drop', ascending=False)
display(importance_df.style.format({'mean_ap_drop': '{:.4f}', 'std': '{:.4f}'}))

ax = importance_df.sort_values('mean_ap_drop').plot.barh(
    x='feature', y='mean_ap_drop', xerr='std', legend=False, figsize=(7, 4),
)
ax.set(xlabel='Decrease in average precision after shuffling', ylabel='', title='Permutation importance on test period')
plt.tight_layout()
plt.show()

## 7. Conclusion

**Yes—there is useful predictive signal for prioritizing accounts, provided `lifetime_months` is excluded.** On the untouched final three months, the leakage-free model should materially beat the prevalence baseline in ROC-AUC, average precision, and top-decile lift. This supports using the score to rank accounts for retention outreach.

The model is not yet a production-ready cancellation probability. Churn prevalence changes sharply across months, the dataset covers fewer than two years, and the holdout has only three months. Before deployment: agree on intervention capacity and error costs; validate across additional future months; monitor ranking and calibration monthly; recalibrate probabilities when needed; and confirm that every feature is available before the scoring cutoff. Do not use the future-derived `lifetime_months` or the account identifier.